# Training

In [ ]:
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

from utils.mused import load_data
from utils.train import get_tokenizer_model, train

In [ ]:
MAX_TOKENS = 256
DROPOUT = 0.0
NAME = "bert_expl_s"
LABELS = "hard"

df_train, df_test = load_data()

model_id = "PlanTL-GOB-ES/roberta-large-bne"
model_id = "BSC-LT/MrBERT-es"

In [ ]:
# tokenizer, model = get_tokenizer_model(model_id)
# res, best_checkpoint = train(df_train, df_test, tokenizer, model, labels=LABELS, max_tokens=MAX_TOKENS)
# print(best_checkpoint)

# Explaining

In [ ]:
best_checkpoint = "baseline_es/checkpoint-20"
tokenizer, model = get_tokenizer_model(model_id, checkpoint=best_checkpoint)

In [ ]:
texts = df_test.text_clean.to_list()

## transformers-interpret

In [ ]:
from transformers_interpret import SequenceClassificationExplainer

cls_explainer = SequenceClassificationExplainer(model, tokenizer)

In [ ]:
word_attributions = cls_explainer(texts[0], class_name="LABEL_1")
cls_explainer.visualize()
print()

## Captum

In [ ]:
from captum.attr import InputXGradient, IntegratedGradients, Saliency

from explain_captum import visualize_attributions

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
print()

In [ ]:
def get_embeddings(input_ids):
    return model.model.embeddings(input_ids)


def forward_fn(embeddings, attention_mask):
    outputs = model(inputs_embeds=embeddings, attention_mask=attention_mask)
    # return logit of predicted class (or a specific class)
    return outputs.logits[:, 1]  # class index 1

In [ ]:
# --- Tokenize ---
text = texts[0]
inputs = tokenizer(text, return_tensors="pt")
input_ids = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)

# --- Get embeddings and baseline (zeros = padding baseline) ---
embeddings = get_embeddings(input_ids)
baseline = torch.zeros_like(embeddings)

In [ ]:
# --- Run Integrated Gradients ---
ig = IntegratedGradients(forward_fn)

attributions, delta = ig.attribute(
    embeddings, baselines=baseline, additional_forward_args=(attention_mask,), return_convergence_delta=True
)

# --- Summarize per token (mean across embedding dim) ---
attributions_sum = attributions.squeeze(0).sum(dim=-1)
tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0))

In [ ]:
# --- Print results ---
for token, score in zip(tokens, attributions_sum.tolist()):
    print(f"{token:20s} {score:.4f}")

In [ ]:
from captum.attr import visualization as viz

# Sum attributions across embedding dim, normalize
attributions_sum = attributions.squeeze(0).sum(dim=-1)
attributions_norm = attributions_sum / torch.norm(attributions_sum)
attributions_norm = attributions_norm.detach().cpu().numpy()

tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0))

# Get prediction info
with torch.no_grad():
    logits = model(input_ids, attention_mask=attention_mask).logits
    pred_class = logits.argmax(dim=-1).item()
    pred_prob = torch.softmax(logits, dim=-1).max().item()

# Build visualization record
vis_record = viz.VisualizationDataRecord(
    word_attributions=attributions_norm,
    pred_prob=pred_prob,
    pred_class=str(pred_class),
    true_class="?",  # put real label if known
    attr_class=str(pred_class),
    attr_score=attributions_norm.sum(),
    raw_input_ids=tokens,
    convergence_score=delta,
)

# Render in notebook
viz.visualize_text([vis_record])
print()

In [ ]:
scores = attributions.squeeze(0).sum(dim=-1).detach().cpu().numpy()
visualize_attributions(tokens, scores)

In [ ]:
import json

with open("caputum_results.json", "r") as f:
    file = json.load(f)

method = "ig"
tokens = file[method]["tokens"]
scores = file[method]["scores"]
words = file[method]["words"]
word_scores = file[method]["word_scores"]
visualize_attributions(tokens, scores)

In [ ]:
visualize_attributions(words, word_scores)

## method suite